# Gráficos — métricas Mininet

Este notebook lê os CSV em `metrics/runs/`, **exibe os gráficos aqui** e salva cópias em `metrics/plots/*.png`.

## O que cada figura responde

### Labs 1 e 2 (topologia `h1 — s1 — h2`)

| Arquivo | Pergunta que o gráfico responde |
|---------|----------------------------------|
| `labs12_throughput_por_tcp_cc.png` | Como **vazão TCP** muda entre cenários (`base`, `delay`, `loss`…) e entre algoritmos (`cubic`, `reno`)? |
| `labs12_rtt_por_tcp_cc.png` | Como o **atraso (RTT)** medido no ping varia nos mesmos cenários e algoritmos? |
| `labs12_perda_por_tcp_cc.png` | Como a **perda de pacotes** no ping varia? (útil quando há `loss` no `tc`) |
| `labs12_painel_tcp_cc.png` | Visão **resumida**: as três métricas lado a lado, para comparar rapidamente |

**Eixo X (cenário):** parâmetros de emulação do enlace (`bw`, `delay`, `loss`) ou `base` se não houve `tc`.  
**Cores (hue):** algoritmo TCP ativo na coleta (`tcp_congestion_control`).

### Lab 3 (roteamento `h1 — r1 — r2 — h2`)

| Arquivo | Pergunta que o gráfico responde |
|---------|----------------------------------|
| `lab3_throughput_direcoes.png` | A **vazão TCP** fim-a-fim é parecida em **h1→h2** e **h2→h1**? Muda quando o **backbone** (`r1↔r2`) está degradado? |
| `lab3_rtt_direcoes.png` | O **RTT** nas duas direções reflete atraso no backbone (`core-delay`)? |
| `lab3_throughput_por_tcp_cc.png` | (Opcional) Com vários algoritmos TCP, quem entrega mais vazão **h1→h2** por cenário? |
| `lab3_rtt_por_tcp_cc.png` | (Opcional) RTT **h1→h2** por cenário e algoritmo TCP |

**Cenário `base`:** backbone sem `--core-bw/delay/loss`.  
**Cenário degradado:** emulação só no link `r1↔r2` (ex.: `bw=10Mbps, delay=30ms, loss=2%`).

In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

%matplotlib inline

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path("/workspace") if Path("/workspace").is_dir() else Path.cwd().parent
RUNS_DIR = ROOT / "metrics" / "runs"
PLOTS_DIR = ROOT / "metrics" / "plots"
CSV_LABS_12 = RUNS_DIR / "labs_1_2_runs.csv"
CSV_LAB3 = RUNS_DIR / "lab3_runs.csv"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

if not CSV_LABS_12.exists() and list(RUNS_DIR.glob("run_*.json")):
    subprocess.run(["python3", str(ROOT / "scripts/json_runs_to_csv.py")], check=False)

print("ROOT:", ROOT)
print("CSV Labs 1-2:", CSV_LABS_12, "existe:", CSV_LABS_12.exists())
print("CSV Lab 3:", CSV_LAB3, "existe:", CSV_LAB3.exists())

In [ ]:
def scenario_labs12(row: pd.Series) -> str:
    parts = []
    if pd.notna(row.get("link_bw_mbps")):
        parts.append(f"bw={row['link_bw_mbps']}Mbps")
    if pd.notna(row.get("link_delay")) and str(row.get("link_delay")).strip():
        parts.append(f"delay={row['link_delay']}")
    if pd.notna(row.get("link_loss_percent")):
        parts.append(f"loss={row['link_loss_percent']}%")
    return ", ".join(parts) if parts else "base"


def scenario_lab3(row: pd.Series) -> str:
    parts = []
    if pd.notna(row.get("core_bw_mbps")):
        parts.append(f"bw={row['core_bw_mbps']}Mbps")
    if pd.notna(row.get("core_delay")) and str(row.get("core_delay")).strip():
        parts.append(f"delay={row['core_delay']}")
    if pd.notna(row.get("core_loss_percent")):
        parts.append(f"loss={row['core_loss_percent']}%")
    return ", ".join(parts) if parts else "base"


def _num(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def show_and_save(fig: plt.Figure, path: Path, dpi: int = 150) -> Path:
    """Exibe a figura no notebook e grava PNG em metrics/plots/."""
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    try:
        from IPython.display import display

        display(fig)
    except Exception:
        plt.show()
    plt.close(fig)
    print("Salvo:", path)
    return path


def plot_cc_comparison(
    df: pd.DataFrame,
    scenario_col: str,
    metric: str,
    ylabel: str,
    title: str,
    filename: str,
    explanation: str = "",
) -> Path | None:
    sub = df.dropna(subset=[metric, "tcp_congestion_control"]).copy()
    if sub.empty:
        print(f"Sem dados para {filename}")
        return None
    if sub["tcp_congestion_control"].nunique() < 1:
        print(f"Sem tcp_congestion_control para {filename}")
        return None

    if explanation:
        print(explanation)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=sub,
        x=scenario_col,
        y=metric,
        hue="tcp_congestion_control",
        ax=ax,
        palette="Set2",
    )
    ax.set_title(title)
    ax.set_xlabel("Cenário de emulação do enlace")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=25)
    ax.legend(title="Algoritmo TCP", bbox_to_anchor=(1.02, 1), loc="upper left")
    return show_and_save(fig, PLOTS_DIR / filename)

## Labs 1 e 2 — comparação por algoritmo de congestionamento

Topologia: `h1 — s1 — h2`. Cada barra agrupa uma coleta (`collect-metrics`).

**Como ler:** barras mais altas em throughput = melhor vazão; em RTT/perda, compare o efeito de `delay` e `loss` e a diferença entre `cubic` e `reno`.

In [ ]:
saved_labs12: list[Path] = []

if CSV_LABS_12.exists():
    df12 = pd.read_csv(CSV_LABS_12)
    df12 = _num(
        df12,
        [
            "iperf_mbits_per_second",
            "rtt_avg_ms",
            "packet_loss_percent",
            "pingall_loss_percent",
            "link_bw_mbps",
            "link_loss_percent",
        ],
    )
    df12["cenario"] = df12.apply(scenario_labs12, axis=1)
    df12["tcp_congestion_control"] = df12["tcp_congestion_control"].fillna("desconhecido")

    display(df12[["cenario", "tcp_congestion_control", "iperf_mbits_per_second", "rtt_avg_ms", "packet_loss_percent"]])

    labs12_plots = [
        (
            "iperf_mbits_per_second",
            "Throughput (Mbit/s)",
            "labs12_throughput_por_tcp_cc.png",
            "Vazão TCP medida pelo iperf3. Mostra quanto tráfego útil passa entre h1 e h2 "
            "em cada cenário; costuma cair com delay/loss e pode diferir entre cubic e reno.",
        ),
        (
            "rtt_avg_ms",
            "RTT médio (ms)",
            "labs12_rtt_por_tcp_cc.png",
            "Tempo de ida e volta médio no ping. Aumenta quando há delay no tc; "
            "reflete latência percebida pelo protocolo antes do TCP ajustar a janela.",
        ),
        (
            "packet_loss_percent",
            "Perda ping (%)",
            "labs12_perda_por_tcp_cc.png",
            "Percentual de pacotes ICMP perdidos no ping h1→h2. "
            "Deve subir quando o cenário inclui loss artificial no enlace.",
        ),
    ]
    for metric, ylabel, fname, expl in labs12_plots:
        p = plot_cc_comparison(
            df12,
            "cenario",
            metric,
            ylabel,
            f"Labs 1-2 — {ylabel}",
            fname,
            explanation=expl,
        )
        if p:
            saved_labs12.append(p)

    # Painel resumo (3 métricas)
    print(
        "Painel resumo: mesma informação dos três gráficos anteriores em uma única figura "
        "(útil para slides e relatório)."
    )
    metrics = [
        ("iperf_mbits_per_second", "Throughput (Mbit/s)"),
        ("rtt_avg_ms", "RTT médio (ms)"),
        ("packet_loss_percent", "Perda ping (%)"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (metric, ylabel) in zip(axes, metrics):
        sub = df12.dropna(subset=[metric])
        if sub.empty:
            ax.set_visible(False)
            continue
        sns.barplot(
            data=sub,
            x="cenario",
            y=metric,
            hue="tcp_congestion_control",
            ax=ax,
            palette="Set2",
        )
        ax.set_title(ylabel)
        ax.set_xlabel("Cenário")
        ax.tick_params(axis="x", rotation=20)
        if ax is axes[0]:
            ax.legend(title="TCP", fontsize=8)
        else:
            leg = ax.get_legend()
            if leg is not None:
                leg.remove()
    fig.suptitle("Labs 1-2 — painel: throughput, RTT e perda", y=1.05)
    saved_labs12.append(show_and_save(fig, PLOTS_DIR / "labs12_painel_tcp_cc.png"))
else:
    print("Arquivo não encontrado:", CSV_LABS_12)
    print("Rode collect-metrics (Labs 1/2) e depois: python3 scripts/json_runs_to_csv.py")

## Lab 3 — roteamento (throughput e RTT)

Topologia: `h1 — r1 — r2 — h2`. O **cenário** descreve o backbone (`r1↔r2`): `base` ou parâmetros `--core-bw`, `--core-delay`, `--core-loss`.

**Como ler:** compare `base` vs cenário degradado; verifique se h1→h2 e h2→h1 são simétricos (barras da mesma altura).

In [ ]:
saved_lab3: list[Path] = []

if CSV_LAB3.exists():
    df3 = pd.read_csv(CSV_LAB3)
    df3 = _num(
        df3,
        [
            "iperf_h1_to_h2_mbits_per_second",
            "iperf_h2_to_h1_mbits_per_second",
            "ping_h1_to_h2_rtt_avg_ms",
            "ping_h2_to_h1_rtt_avg_ms",
            "core_bw_mbps",
            "core_loss_percent",
        ],
    )
    df3["cenario"] = df3.apply(scenario_lab3, axis=1)
    if "tcp_congestion_control" in df3.columns:
        df3["tcp_congestion_control"] = df3["tcp_congestion_control"].fillna("desconhecido")

    # Throughput bidirecional
    melt_iperf = df3.melt(
        id_vars=["cenario", "tcp_congestion_control"] if "tcp_congestion_control" in df3.columns else ["cenario"],
        value_vars=["iperf_h1_to_h2_mbits_per_second", "iperf_h2_to_h1_mbits_per_second"],
        var_name="direcao",
        value_name="mbits_per_second",
    )
    melt_iperf["direcao"] = melt_iperf["direcao"].str.replace("iperf_", "").str.replace("_mbits_per_second", "")

    print(
        "Throughput por direção: vazão TCP medida com servidor em um extremo e cliente no outro. "
        "Avalia se o gargalo no backbone afeta igualmente os dois sentidos."
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=melt_iperf.dropna(subset=["mbits_per_second"]),
        x="cenario",
        y="mbits_per_second",
        hue="direcao",
        ax=ax,
        palette="Set2",
    )
    ax.set_title("Lab 3 — throughput TCP por direção")
    ax.set_xlabel("Cenário do backbone (r1↔r2)")
    ax.set_ylabel("Mbit/s")
    ax.legend(title="Direção")
    ax.tick_params(axis="x", rotation=25)
    saved_lab3.append(show_and_save(fig, PLOTS_DIR / "lab3_throughput_direcoes.png"))

    # RTT bidirecional
    melt_rtt = df3.melt(
        id_vars=["cenario"],
        value_vars=["ping_h1_to_h2_rtt_avg_ms", "ping_h2_to_h1_rtt_avg_ms"],
        var_name="direcao",
        value_name="rtt_avg_ms",
    )
    melt_rtt["direcao"] = melt_rtt["direcao"].str.replace("ping_", "").str.replace("_rtt_avg_ms", "")

    print(
        "RTT por direção: latência média do ping em cada sentido. "
        "Com core-delay no backbone, espera-se RTT maior que no cenário base."
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=melt_rtt.dropna(subset=["rtt_avg_ms"]),
        x="cenario",
        y="rtt_avg_ms",
        hue="direcao",
        ax=ax,
        palette="Set2",
    )
    ax.set_title("Lab 3 — RTT médio por direção")
    ax.set_xlabel("Cenário do backbone (r1↔r2)")
    ax.set_ylabel("ms")
    ax.legend(title="Direção")
    ax.tick_params(axis="x", rotation=25)
    saved_lab3.append(show_and_save(fig, PLOTS_DIR / "lab3_rtt_direcoes.png"))

    display(
        df3[
            [
                "cenario",
                "tcp_congestion_control",
                "iperf_h1_to_h2_mbits_per_second",
                "iperf_h2_to_h1_mbits_per_second",
                "ping_h1_to_h2_rtt_avg_ms",
                "ping_h2_to_h1_rtt_avg_ms",
            ]
        ].drop_duplicates()
        if "tcp_congestion_control" in df3.columns
        else df3[
            [
                "cenario",
                "iperf_h1_to_h2_mbits_per_second",
                "iperf_h2_to_h1_mbits_per_second",
                "ping_h1_to_h2_rtt_avg_ms",
                "ping_h2_to_h1_rtt_avg_ms",
            ]
        ]
    )

    # Comparação tcp_cc no Lab 3 (se houver mais de um algoritmo nas coletas)
    if "tcp_congestion_control" in df3.columns and df3["tcp_congestion_control"].nunique() > 1:
        lab3_cc_plots = [
            (
                "iperf_h1_to_h2_mbits_per_second",
                "Throughput h1→h2 (Mbit/s)",
                "lab3_throughput_por_tcp_cc.png",
                "Compara algoritmos TCP (cubic vs reno) na vazão h1→h2, por cenário de backbone.",
            ),
            (
                "ping_h1_to_h2_rtt_avg_ms",
                "RTT h1→h2 (ms)",
                "lab3_rtt_por_tcp_cc.png",
                "RTT no sentido h1→h2 por algoritmo TCP e cenário.",
            ),
        ]
        for metric, ylabel, fname, expl in lab3_cc_plots:
            p = plot_cc_comparison(
                df3,
                "cenario",
                metric,
                ylabel,
                f"Lab 3 — {ylabel}",
                fname,
                explanation=expl,
            )
            if p:
                saved_lab3.append(p)
    elif "tcp_congestion_control" in df3.columns:
        print(
            "Gráficos lab3_*_por_tcp_cc omitidos: há apenas um algoritmo TCP nas coletas. "
            "Rode collect-metrics-routing com cubic e reno para comparar."
        )
else:
    print("Arquivo não encontrado:", CSV_LAB3)
    print("Rode collect-metrics-routing (Lab 3) e depois: python3 scripts/json_runs_to_csv.py")

In [ ]:
print("\n=== Resumo ===")
print("Labs 1-2:", len(saved_labs12), "figura(s)")
for p in saved_labs12:
    print(" ", p.name)
print("Lab 3:", len(saved_lab3), "figura(s)")
for p in saved_lab3:
    print(" ", p.name)
print("Pasta PNG:", PLOTS_DIR)
print("\nDica: abra este notebook com ./run.sh plots-lab para ver os gráficos interativamente.")